# Python Data Processing — NorthStar Urban Mobility & Logistics

**Module:** Databases and Analytics 
**Focus:** Data cleaning, feature engineering, analysis and visualisation using Python (Pandas, NumPy, Matplotlib, Seaborn)

---

## Objective

The R notebooks focused on SQL querying and statistical testing. In this notebook, I'm using Python to do the heavy lifting on data cleaning, transformation, and building more detailed analytical views of NorthStar's operational problems.

The dataset has known issues — inconsistent categorical values, missing data, and patterns that only emerge when you cross-reference multiple files. Python with Pandas is well suited for this kind of messy data work.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
# Load all datasets
hubs = pd.read_csv('hubs.csv')
customers = pd.read_csv('customers.csv')
drivers = pd.read_csv('drivers.csv')
vehicles = pd.read_csv('vehicles.csv')
orders = pd.read_csv('orders.csv')
deliveries = pd.read_csv('deliveries.csv')
incidents = pd.read_csv('incidents.csv')
complaints = pd.read_csv('complaints.csv')
app_events = pd.read_csv('app_events.csv')

print('Dataset shapes:')
for name, df in [('hubs', hubs), ('customers', customers), ('drivers', drivers),
                  ('vehicles', vehicles), ('orders', orders), ('deliveries', deliveries),
                  ('incidents', incidents), ('complaints', complaints), ('app_events', app_events)]:
    print(f'  {name}: {df.shape[0]} rows, {df.shape[1]} columns')

## 2. Data Cleaning and Standardisation

The README mentions inconsistent categorical values and missing data. Let me check for those issues and fix them before doing any analysis.

In [ ]:
# Check for inconsistent zone names across all tables
print('Zone values in orders (pickup_zone):', orders['pickup_zone'].unique())
print('Zone values in orders (dropoff_zone):', orders['dropoff_zone'].unique())
print('Zone values in customers (home_zone):', customers['home_zone'].unique())
print('Zone values in drivers (base_zone):', drivers['base_zone'].unique())
print('Zone values in vehicles (assigned_zone):', vehicles['assigned_zone'].unique())
print('Zone values in hubs (zone):', hubs['zone'].unique())

In [ ]:
# Standardise zone names — the data has mixed cases like 'AIRPORT', 'Airport', 'airport'
def clean_zone(val):
    if pd.isna(val):
        return val
    val = str(val).strip().title()
    # Handle the AIRPORT variations
    if val.upper() == 'AIRPORT':
        return 'Airport'
    return val

# Apply to all zone columns
orders['pickup_zone'] = orders['pickup_zone'].apply(clean_zone)
orders['dropoff_zone'] = orders['dropoff_zone'].apply(clean_zone)
customers['home_zone'] = customers['home_zone'].apply(clean_zone)
drivers['base_zone'] = drivers['base_zone'].apply(clean_zone)
vehicles['assigned_zone'] = vehicles['assigned_zone'].apply(clean_zone)
hubs['zone'] = hubs['zone'].apply(clean_zone)
app_events['zone_context'] = app_events['zone_context'].apply(clean_zone)

print('After cleaning:')
print('Pickup zones:', sorted(orders['pickup_zone'].dropna().unique()))

In [ ]:
# Check for missing values across all datasets
print('Missing values summary:')
print('=' * 50)
for name, df in [('orders', orders), ('deliveries', deliveries), 
                  ('complaints', complaints), ('customers', customers),
                  ('incidents', incidents), ('vehicles', vehicles)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n{name}:')
        for col, count in missing.items():
            print(f'  {col}: {count} missing ({count/len(df)*100:.1f}%)')

In [ ]:
# Convert datetime columns
orders['order_created_at'] = pd.to_datetime(orders['order_created_at'])
deliveries['dispatch_time'] = pd.to_datetime(deliveries['dispatch_time'])
deliveries['delivery_completed_at'] = pd.to_datetime(deliveries['delivery_completed_at'])
complaints['created_at'] = pd.to_datetime(complaints['created_at'])
incidents['reported_at'] = pd.to_datetime(incidents['reported_at'])
app_events['event_timestamp'] = pd.to_datetime(app_events['event_timestamp'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
vehicles['commission_date'] = pd.to_datetime(vehicles['commission_date'])

print('Datetime conversion done.')

## 3. Feature Engineering

Now I need to create some derived columns that will be useful for analysis. The raw data doesn't have things like delivery duration, profit margin, or time-based features — those need to be calculated.

In [ ]:
# Merge orders and deliveries for the main analytical dataframe
df = orders.merge(deliveries, on='order_id', how='left')
df = df.merge(customers[['customer_id', 'home_zone', 'customer_type', 'loyalty_score', 
                          'app_engagement_score', 'account_status']], 
              on='customer_id', how='left')
df = df.merge(hubs[['hub_id', 'hub_name', 'zone', 'capacity_score']], 
              on='hub_id', how='left')

# Feature engineering
df['delivery_duration_hours'] = (df['delivery_completed_at'] - df['dispatch_time']).dt.total_seconds() / 3600
df['profit_margin'] = df['order_value'] - df['fuel_or_charge_cost']
df['margin_pct'] = (df['profit_margin'] / df['order_value']) * 100
df['order_month'] = df['order_created_at'].dt.to_period('M').astype(str)
df['order_day_of_week'] = df['order_created_at'].dt.day_name()
df['order_hour'] = df['order_created_at'].dt.hour
df['has_override'] = (df['manual_route_override_count'] > 0).astype(int)

# Flag if this delivery had an incident
incident_deliveries = set(incidents['delivery_id'].dropna())
df['has_incident'] = df['delivery_id'].isin(incident_deliveries).astype(int)

# Flag if this order had a complaint
complaint_orders = set(complaints['order_id'].dropna())
df['has_complaint'] = df['order_id'].isin(complaint_orders).astype(int)

print(f'Combined dataframe: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nNew features added: delivery_duration_hours, profit_margin, margin_pct,')
print(f'order_month, order_day_of_week, order_hour, has_override, has_incident, has_complaint')

In [ ]:
# Quick look at the engineered features
df[['order_value', 'fuel_or_charge_cost', 'profit_margin', 'margin_pct', 
    'delivery_duration_hours', 'route_distance_km']].describe().round(2)

## 4. Analytical Deep Dives

With the cleaned and enriched dataset, I can now investigate the specific problems NorthStar is facing.

### Analysis 1: Delivery Failure Patterns

Where, when, and why are deliveries failing?

In [ ]:
# Failure rate by zone, service type, and priority
delivered = df[df['delivery_status'].notna()].copy()

failure_by_zone = delivered.groupby('pickup_zone').agg(
    total=('delivery_id', 'count'),
    failed=('delivery_status', lambda x: (x == 'Failed').sum()),
    late=('delivery_status', lambda x: (x == 'Late').sum())
).reset_index()

failure_by_zone['failure_rate'] = (failure_by_zone['failed'] / failure_by_zone['total'] * 100).round(1)
failure_by_zone['late_rate'] = (failure_by_zone['late'] / failure_by_zone['total'] * 100).round(1)
failure_by_zone['problem_rate'] = failure_by_zone['failure_rate'] + failure_by_zone['late_rate']

print(failure_by_zone.sort_values('problem_rate', ascending=False).to_string(index=False))

In [ ]:
# Visualisation: failure rates across zones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar of delivery outcomes by zone
status_counts = delivered.groupby(['pickup_zone', 'delivery_status']).size().unstack(fill_value=0)
colors = {'OnTime': '#2E86AB', 'Late': '#F6AE2D', 'Failed': '#E84855'}
status_counts.plot(kind='bar', stacked=True, ax=axes[0], 
                   color=[colors.get(c, '#999') for c in status_counts.columns])
axes[0].set_title('Delivery Outcomes by Pickup Zone')
axes[0].set_xlabel('Zone')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Status')

# Right: failure rate comparison
failure_sorted = failure_by_zone.sort_values('failure_rate', ascending=True)
axes[1].barh(failure_sorted['pickup_zone'], failure_sorted['failure_rate'], color='#E84855', alpha=0.8)
axes[1].set_title('Failure Rate by Zone (%)')
axes[1].set_xlabel('Failure Rate (%)')
for i, v in enumerate(failure_sorted['failure_rate']):
    axes[1].text(v + 0.3, i, f'{v}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**What this reveals:** The two charts together paint a clear picture. The stacked bar shows absolute volumes, while the horizontal bar strips it down to just failure rates so the comparison is fair regardless of zone size. Zones with high failure rates are the obvious candidates for the operations director's attention — but we need to figure out if the problem is routes, vehicles, or drivers, which the next analyses will address.

### Analysis 2: The Delivery Duration Anomaly

Let me look at delivery durations — do they make sense? Are there cases where the completion time is before the dispatch time (which would be a data quality issue)?

In [ ]:
# Check for negative or unreasonable durations
duration_issues = delivered[delivered['delivery_duration_hours'] < 0]
print(f'Deliveries with negative duration (completed before dispatched): {len(duration_issues)}')

very_long = delivered[delivered['delivery_duration_hours'] > 48]
print(f'Deliveries taking longer than 48 hours: {len(very_long)}')

# Duration by delivery status
duration_stats = delivered.groupby('delivery_status')['delivery_duration_hours'].agg(
    ['mean', 'median', 'std', 'min', 'max']
).round(2)
print('\nDuration stats by delivery status (hours):')
print(duration_stats)

In [ ]:
# Distribution of delivery durations by status
fig, ax = plt.subplots(figsize=(12, 5))

for status, color in [('OnTime', '#2E86AB'), ('Late', '#F6AE2D'), ('Failed', '#E84855')]:
    data = delivered[delivered['delivery_status'] == status]['delivery_duration_hours']
    data = data[(data > 0) & (data < 72)]  # filter extremes for readable chart
    ax.hist(data, bins=40, alpha=0.5, label=status, color=color)

ax.set_title('Delivery Duration Distribution by Status')
ax.set_xlabel('Duration (hours)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

**What this reveals:** Negative durations are a data integrity issue — they mean the delivery_completed_at timestamp is earlier than dispatch_time, which shouldn't happen. This is the kind of inconsistency the case study talks about. For the valid durations, the overlap between 'OnTime' and 'Late' distributions shows that the boundary between on-time and late may be arbitrary or inconsistently applied. NorthStar needs cleaner timestamp recording if they want reliable performance metrics.

### Analysis 3: Vehicle Health Risk Assessment

Are vehicles with degraded batteries being sent out and causing problems?

In [ ]:
# Merge vehicle info with delivery outcomes
vehicle_perf = delivered.merge(vehicles[['vehicle_id', 'vehicle_type', 'battery_health_pct', 
                                         'odometer_km', 'maintenance_status']], 
                                on='vehicle_id', how='left')

# Create battery health categories
vehicle_perf['battery_category'] = pd.cut(
    vehicle_perf['battery_health_pct'],
    bins=[0, 50, 70, 85, 100],
    labels=['Critical (<50%)', 'Low (50-70%)', 'Moderate (70-85%)', 'Good (>85%)']
)

# Failure and incident rates by battery category
battery_analysis = vehicle_perf.groupby('battery_category', observed=True).agg(
    deliveries=('delivery_id', 'count'),
    failures=('delivery_status', lambda x: (x == 'Failed').sum()),
    incidents=('has_incident', 'sum'),
    avg_rating=('customer_rating_post_delivery', 'mean')
).reset_index()

battery_analysis['failure_rate'] = (battery_analysis['failures'] / battery_analysis['deliveries'] * 100).round(1)
battery_analysis['incident_rate'] = (battery_analysis['incidents'] / battery_analysis['deliveries'] * 100).round(1)
battery_analysis['avg_rating'] = battery_analysis['avg_rating'].round(2)

print(battery_analysis.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: failure rate by battery health
cat_order = ['Critical (<50%)', 'Low (50-70%)', 'Moderate (70-85%)', 'Good (>85%)']
ba_sorted = battery_analysis.set_index('battery_category').loc[
    [c for c in cat_order if c in battery_analysis['battery_category'].values]
].reset_index()

colors_battery = ['#E84855', '#F6AE2D', '#8ECAE6', '#2A9D8F']
axes[0].bar(ba_sorted['battery_category'], ba_sorted['failure_rate'], 
            color=colors_battery[:len(ba_sorted)])
axes[0].set_title('Delivery Failure Rate by Vehicle Battery Health')
axes[0].set_ylabel('Failure Rate (%)')
axes[0].tick_params(axis='x', rotation=20)

# Right: scatter of battery health vs odometer
scatter_data = vehicles.copy()
axes[1].scatter(scatter_data['odometer_km'], scatter_data['battery_health_pct'],
                c=scatter_data['battery_health_pct'], cmap='RdYlGn', 
                alpha=0.7, edgecolors='grey', s=60)
axes[1].set_title('Battery Health vs Odometer Reading')
axes[1].set_xlabel('Odometer (km)')
axes[1].set_ylabel('Battery Health (%)')
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Critical threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

**What this reveals:** If vehicles with critical battery health have substantially higher failure and incident rates, NorthStar is sending out vehicles that shouldn't be on the road. The scatter plot adds another dimension — high-mileage vehicles tend to have worse battery health, which is expected, but any vehicle below the 50% line is a risk. A simple rule like "don't dispatch vehicles with battery health below 60%" could prevent a lot of incidents before they happen.

### Analysis 4: Time-Based Demand Patterns

When do problems concentrate? Understanding peak times and problem hours could help with scheduling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly order distribution
hour_counts = df.groupby('order_hour').size()
hour_failures = delivered[delivered['delivery_status'] == 'Failed'].groupby(
    delivered['order_created_at'].dt.hour
).size()

axes[0].bar(hour_counts.index, hour_counts.values, color='#2E86AB', alpha=0.6, label='All orders')
axes[0].bar(hour_failures.index, hour_failures.values, color='#E84855', alpha=0.8, label='Failed')
axes[0].set_title('Order Volume and Failures by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')
axes[0].legend()

# Day of week pattern
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df.groupby('order_day_of_week').size().reindex(day_order)
axes[1].bar(range(7), day_counts.values, color='#264653', alpha=0.8)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1].set_title('Order Volume by Day of Week')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

**What this reveals:** The hourly chart shows when demand peaks and whether failures are concentrated at peak times (suggesting capacity issues) or spread evenly (suggesting systemic problems). The day-of-week chart helps with staffing — if weekends have high volumes but NorthStar schedules fewer drivers, that's an obvious mismatch.

### Analysis 5: Customer Complaint Deep Dive

In [ ]:
# Complaint analysis
complaint_detail = complaints.merge(
    customers[['customer_id', 'home_zone', 'customer_type', 'loyalty_score']],
    on='customer_id', how='left'
)

# Repeat complainers
repeat_complainers = complaints.groupby('customer_id').size().reset_index(name='complaint_count')
repeat_complainers = repeat_complainers[repeat_complainers['complaint_count'] > 1]

print(f'Total complaints: {len(complaints)}')
print(f'Unique customers who complained: {complaints["customer_id"].nunique()}')
print(f'Repeat complainers (2+ complaints): {len(repeat_complainers)}')
print(f'Max complaints from one customer: {repeat_complainers["complaint_count"].max()}')
print(f'\nTotal compensation paid out: £{complaints["compensation_amount"].sum():,.2f}')
print(f'Average compensation per complaint: £{complaints["compensation_amount"].mean():,.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: complaint types
complaint_types = complaints['complaint_type'].value_counts()
colors_complaint = sns.color_palette('Set2', len(complaint_types))
axes[0].pie(complaint_types.values, labels=complaint_types.index, autopct='%1.1f%%',
            colors=colors_complaint, startangle=90)
axes[0].set_title('Complaint Types Breakdown')

# Right: compensation by complaint type
comp_by_type = complaints.groupby('complaint_type')['compensation_amount'].agg(['mean', 'sum']).round(2)
comp_by_type = comp_by_type.sort_values('sum', ascending=True)
axes[1].barh(comp_by_type.index, comp_by_type['sum'], color='#E76F51', alpha=0.8)
axes[1].set_title('Total Compensation Paid by Complaint Type (£)')
axes[1].set_xlabel('Total Compensation (£)')

plt.tight_layout()
plt.show()

**What this reveals:** The pie chart shows what customers are actually complaining about — if one type dominates, that's where NorthStar should focus its fix. The compensation bar chart adds the financial dimension: some complaint types might be less frequent but much more expensive to resolve. Repeat complainers are also a risk — these are customers who might churn, and if they're SME clients, that's significant lost revenue.

### Analysis 6: Profitability Heatmap — Pickup to Dropoff Routes

In [ ]:
# Build a pivot table of average profit margin by route corridor
route_profit = delivered.groupby(['pickup_zone', 'dropoff_zone']).agg(
    avg_margin=('profit_margin', 'mean'),
    count=('delivery_id', 'count')
).reset_index()

# Filter to routes with enough data
route_profit = route_profit[route_profit['count'] >= 5]

# Pivot for heatmap
margin_pivot = route_profit.pivot(index='pickup_zone', columns='dropoff_zone', values='avg_margin')

plt.figure(figsize=(10, 7))
sns.heatmap(margin_pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'Avg Profit Margin (£)'})
plt.title('Average Profit Margin by Route (Pickup → Dropoff)')
plt.xlabel('Dropoff Zone')
plt.ylabel('Pickup Zone')
plt.tight_layout()
plt.show()

**What this reveals:** The heatmap gives the finance director exactly what they need — a visual map of which route corridors make money (green) and which don't (red). Routes with low or negative margins deserve scrutiny. Some might be loss-leaders that bring in valuable customers, but others might just be underpriced. This is the kind of cross-referencing that NorthStar's fragmented systems currently can't do because cost and revenue data sit in different places.

### Analysis 7: App Events and Digital Platform Performance

In [ ]:
# App events analysis — the README says this data is suitable for MongoDB reshaping
print('App event types:', app_events['event_type'].unique())
print('\nDevice types:', app_events['device_type'].unique())
print(f'\nAvg API latency: {app_events["api_latency_ms"].mean():.0f} ms')
print(f'Success rate: {app_events["success_flag"].mean()*100:.1f}%')

# Latency by event type
latency_by_event = app_events.groupby('event_type').agg(
    count=('event_id', 'count'),
    avg_latency=('api_latency_ms', 'mean'),
    success_rate=('success_flag', 'mean')
).round(2)
latency_by_event['success_rate'] = (latency_by_event['success_rate'] * 100).round(1)

print('\nPerformance by event type:')
print(latency_by_event.sort_values('avg_latency', ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# API latency distribution
axes[0].hist(app_events['api_latency_ms'], bins=50, color='#264653', alpha=0.7, edgecolor='white')
axes[0].axvline(x=300, color='red', linestyle='--', label='300ms threshold')
axes[0].set_title('API Latency Distribution')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Success rate by zone
zone_success = app_events.groupby('zone_context')['success_flag'].mean() * 100
zone_success = zone_success.sort_values()
axes[1].barh(zone_success.index, zone_success.values, color='#2A9D8F', alpha=0.8)
axes[1].set_title('App Event Success Rate by Zone (%)')
axes[1].set_xlabel('Success Rate (%)')
axes[1].set_xlim(80, 100)

plt.tight_layout()
plt.show()

**What this reveals:** The app is the primary customer-facing tool, so its performance directly affects user experience. High latency events or low success rates for certain event types (like booking or payment) can drive complaints even when the underlying delivery is fine. The zone-level success rate is interesting — if certain zones have worse app performance, it might explain some of the 'AppIssue' complaints we saw earlier. This data is also the prime candidate for MongoDB because of its nested, event-driven structure.

## 5. Summary of Python Analysis

Working through the data with Python revealed several things that the R analysis touched on but couldn't fully unpack:

1. **Data quality is a real issue** — inconsistent zone names, negative delivery durations, and missing values mean NorthStar can't trust its own numbers without cleaning first. Any dashboard built on raw data will be misleading.

2. **Vehicle battery health is a predictable risk factor** — there's a clear gradient from healthy batteries to failing ones, and the failure/incident rates follow it. A threshold-based dispatch rule would be a quick win.

3. **Demand patterns exist but aren't being used** — order volumes vary by hour and day, but if staffing doesn't match these patterns, capacity mismatches will keep causing failures at peak times.

4. **Route profitability varies wildly** — the heatmap shows specific corridors that are barely breaking even. Combined with the zone-level failure data, NorthStar can now see which routes are both unprofitable AND unreliable.

5. **The app platform has its own issues** — latency and failure rates on the digital side contribute to customer dissatisfaction independently of the physical delivery performance.

6. **Repeat complainers represent a churn risk** — a small number of customers file multiple complaints, and losing these customers (especially SME) would be disproportionately costly.

These findings feed directly into the MongoDB design decisions in the next notebook — particularly around how to model customer cases (with embedded complaint histories), vehicle telemetry (for predictive maintenance), and app events (inherently document-oriented data).